In [6]:
%pip install anomalib
!anomalib install -v
from anomalib.data import MVTec
from anomalib.models import Patchcore, EfficientAd
from anomalib import TaskType
from anomalib.engine import Engine
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

Note: you may need to restart the kernel to use updated packages.
/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/anomalib/cli/install.py:63: UserWarning: Torch Version will be selected as 2.2.0.
  torch_install_args = get_torch_install_args(torch_requirement)
/bin/sh: 1: nvcc: not found
⠋ Installing packages...  This may take a 

### MVTec Dataset
The MVTec Anomaly Detection (MVTec-AD) dataset is a widely used benchmark for evaluating anomaly detection methods, particularly in industrial quality inspection contexts. It contains high-resolution images across 15 object categories, such as "tile," "leather," "bottle," and "grid," each with a variety of normal (defect-free) and anomalous (defective) samples. Anomalies include visual defects like scratches, dents, or incorrect textures, which make the dataset ideal for testing algorithms on diverse real-world defect types.


In [7]:
task = TaskType.DETECTION
categories = ["tile", "leather", "grid"]

datasets = []
for category in categories:
    datamodule = MVTec(category=category,task=task)
    datamodule.prepare_data()
    datamodule.setup()
    datasets.append(datamodule)

callbacks = [
    ModelCheckpoint(
        mode="max",
        monitor="image_AUROC",
    ),
    EarlyStopping(
        monitor="image_AUROC",
        mode="max",
        patience=3,
    ),
]

### Patchcore
PatchCore is an anomaly detection model included in the anomalib library, tailored to identify anomalies in image data by analyzing local patches instead of whole images. This patch-based approach is particularly useful for datasets like MVTec-AD, where defects or anomalies are often subtle and confined to small regions.

#### How PatchCore Works
PatchCore extracts local patches from images, creating a high-dimensional feature representation of each patch. During training, it models these patches’ normal patterns to build a baseline of "normality." When evaluating new data, PatchCore identifies deviations from this normal baseline, flagging patches that look significantly different as potential anomalies.

#### The Role of Coresets
To efficiently manage high-dimensional patch data, PatchCore uses coresets. A coreset is a small, representative subset of the training patches that effectively capture the variability in normal images. Instead of using the full dataset of patches, which could be computationally overwhelming, PatchCore relies on these coresets. This reduces memory usage and speeds up inference without losing essential information on normal patterns. By comparing patches from test images to this representative subset, the model can detect anomalies by identifying patches that diverge from the coreset distribution.

In summary, coresets in PatchCore make the model more efficient, enabling it to focus only on the most informative patches while preserving the ability to accurately identify anomalies.

In [8]:
patchcore_model = Patchcore(pre_trained=True)

engine = Engine(task=task, callbacks=callbacks, pixel_metrics='AUROC')

category_level_auroc = {}
for category, dataset in zip(categories, datasets):
    engine.fit(datamodule=dataset, model=patchcore_model)
    results = engine.test(datamodule=dataset, model=patchcore_model)
    category_level_auroc[category] = results[0]['image_AUROC']

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/lightning/pytorch/core/optimizer.py:182: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                     ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model                 │ PatchcoreModel           │ 24.9 M │ train │
│ 1 │ _transform            │ Compose                  │      0 │ train │
│ 2 │ normalization_metrics │ MetricCollection         │      0 │ train │
│ 3 │ image_threshold       │ F1AdaptiveThreshold      │      0 │ train │
│ 4 │ pixel_threshold       │ F1AdaptiveThreshold      │      0 │ train │
│ 5 │ image_metrics         │ AnomalibMetricCollection │      0 │ train │
│ 6 │ pixel_metrics         │ AnomalibMetricCollection │      0 │ train │
└───┴───────────────────────┴──────────────────────────┴────────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99                                                                         
Modules in train mode: 15                                                                                          
Modules in eval mode: 174

Output()

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/lightning/pytorch/loops/optimization/automatic.py:132:
`training_step` returned `None`. If this was on purpose, ignore this warning...

Output()

... (29 outputs left out)

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99                                                                         
Modules in train mode: 15                                                                                          
Modules in eval mode: 174

`Trainer.fit` stopped: `max_epochs=1` reached.
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            0.5            │
│       image_F1Score       │    0.8444444537162781     │
└───────────────────────────┴───────────────────────────┘

In [43]:
average_auroc = sum(category_level_auroc.values()) / len(categories)
category_level_auroc['Average'] = average_auroc
#print(category_level_auroc)
for category, auroc in category_level_auroc.items():
    print(f"{category}: {auroc}")


image_path = "./datasets/MVTec/grid/test/broken/000.png"

pred = engine.predict(data_path=image_path, model=patchcore_model)

#print(pred)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Load the original image and get its dimensions
original_image = Image.open("./datasets/MVTec/grid/test/broken/000.png")
original_width, original_height = original_image.size

# Convert and resize the anomaly maps to match the original image dimensions
anomaly_maps = [
    np.squeeze(anomaly_map.cpu().numpy())
    for anomaly_map in pred[0]["anomaly_maps"]
]
anomaly_maps_resized = [
    np.array(Image.fromarray((anomaly_map * 255).astype(np.uint8)).resize((original_width, original_height)))
    for anomaly_map in anomaly_maps
]

# Define a threshold to show only the high-anomaly areas
threshold = 0.5 * 255

# Plot the results
num_maps = len(anomaly_maps_resized)
fig, ax = plt.subplots(1, num_maps + 1, figsize=(6 * (num_maps + 1), 6))

# Original image
ax[0].imshow(original_image)
ax[0].set_title("Original Image")
ax[0].axis("off")

# Only highlight high-anomaly areas in each map
for i, anomaly_map_resized in enumerate(anomaly_maps_resized):
    # Create a mask for high anomaly areas
    anomaly_mask = anomaly_map_resized > threshold
    high_anomaly_areas = np.zeros_like(anomaly_map_resized)
    high_anomaly_areas[anomaly_mask] = anomaly_map_resized[anomaly_mask]

    ax[i + 1].imshow(original_image)
    ax[i + 1].imshow(high_anomaly_areas, cmap="jet", alpha=0.5)
    ax[i + 1].set_title(f"Anomaly Areas {i + 1}")
    ax[i + 1].axis("off")

# Display
plt.tight_layout()
output_path = "./high_anomaly_areas_result.png"
plt.savefig(output_path)
print(f"High anomaly areas results saved to {output_path}")






ckpt_path is not provided. Model weights will not be loaded.
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


tile: 0.9862914681434631
leather: 0.6229619979858398
grid: 0.5
Average: 1.0546267330646515


Output()

High anomaly areas results saved to ./high_anomaly_areas_result.png


### EfficientAD
EfficientAD is another anomaly detection model in the anomalib library designed for high efficiency and accuracy, especially in environments where computational resources might be limited. EfficientAD is tailored for applications like industrial defect detection, where anomalies can be subtle and vary greatly in appearance.

#### How EfficientAD Works
EfficientAD builds on efficient feature extraction methods to create a lightweight and fast anomaly detection pipeline. Unlike models that analyze entire images or large volumes of data, EfficientAD focuses on extracting meaningful features in a resource-efficient way, often by leveraging lightweight architectures and techniques like convolutional neural networks (CNNs). During training, EfficientAD learns patterns and representations of the normal data by analyzing these compact features.

#### Lightweight Feature Representations
EfficientAD is designed to optimize both memory usage and inference speed. It does this by creating smaller feature representations of images, which contain the essential information needed to identify anomalies while avoiding unnecessary computational overhead. This makes EfficientAD especially useful for real-time applications, where speed and resource efficiency are critical.

#### Detecting Anomalies
When a new image is presented, EfficientAD compares the feature representation of the image to the learned distribution of normal images. Deviations from this distribution are flagged as anomalies. By focusing on compact and efficient representations, EfficientAD can maintain high accuracy while being less computationally intensive compared to other models.

In summary, EfficientAD’s use of lightweight feature extraction and efficient representations allows it to perform accurate anomaly detection with minimal computational demands, making it an ideal choice for scenarios requiring quick and resource-conscious processing.

In [46]:
efficient_ad_model = EfficientAd()
engine = Engine(task=task)


category_level_auroc = {}
for category, dataset in zip(categories, datasets):
    dataset.train_batch_size = 1
    engine.fit(datamodule=dataset, model=efficient_ad_model)  # Train EfficientAD
    results = engine.test(datamodule=dataset, model=efficient_ad_model)  # Test EfficientAD
    category_level_auroc[category] = results[0]["image_AUROC"]


Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/lightning/pytorch/loops/utilities.py:72: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                     ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model                 │ EfficientAdModel         │  8.1 M │ train │
│ 1 │ _transform            │ Compose                  │      0 │ train │
│ 2 │ normalization_metrics │ MetricCollection         │      0 │ train │
│ 3 │ image_threshold       │ F1AdaptiveThreshold      │      0 │ train │
│ 4 │ pixel_threshold       │ F1AdaptiveThreshold      │      0 │ train │
│ 5 │ image_metrics         │ AnomalibMetricCollection │      0 │ train │
│ 6 │ pixel_metrics         │ AnomalibMetricCollection │      0 │ train │
└───┴───────────────────────┴──────────────────────────┴────────┴───────┘

Trainable params: 8.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.1 M                                                                                                
Total estimated model params size (MB): 32                                                                         
Modules in train mode: 44                                                                                          
Modules in eval mode: 7

Output()

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/anomalib/models/image/efficient_ad/lightning_model.py:
98: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle
module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during 
unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a 
future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that 
could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless 
they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start 
setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an 
issue on GitHub for any issues related to this experimental feature.

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: 
UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use 
`v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(

Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/230 [00:00<?, ?it/s]
Calculate teacher chan

RecursionError: maximum recursion depth exceeded in comparison

In [ ]:
average_auroc = sum(category_level_auroc.values()) / len(categories)
category_level_auroc['Average'] = average_auroc
#print(category_level_auroc)
for category, auroc in category_level_auroc.items():
    print(f"{category}: {auroc}")


In [45]:

datamodule = MVTec(train_batch_size=1, category='grid', task=TaskType.DETECTION)
datamodule.prepare_data()
datamodule.setup()
model = EfficientAd()
engine = Engine()

# Train the model
engine.train(datamodule=datamodule, model=model)

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/lightning/pytorch/loops/utilities.py:72: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                     ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model                 │ EfficientAdModel         │  8.1 M │ train │
│ 1 │ _transform            │ Compose                  │      0 │ train │
│ 2 │ normalization_metrics │ MetricCollection         │      0 │ train │
│ 3 │ image_threshold       │ F1AdaptiveThreshold      │      0 │ train │
│ 4 │ pixel_threshold       │ F1AdaptiveThreshold      │      0 │ train │
│ 5 │ image_metrics         │ AnomalibMetricCollection │      0 │ train │
│ 6 │ pixel_metrics         │ AnomalibMetricCollection │      0 │ train │
└───┴───────────────────────┴──────────────────────────┴────────┴───────┘

Trainable params: 8.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.1 M                                                                                                
Total estimated model params size (MB): 32                                                                         
Modules in train mode: 46                                                                                          
Modules in eval mode: 7

Output()

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/anomalib/models/image/efficient_ad/lightning_model.py:
98: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle
module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during 
unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a 
future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that 
could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless 
they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start 
setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an 
issue on GitHub for any issues related to this experimental feature.

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: 
UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use 
`v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(

Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher channel mean & std:   0%|          | 0/264 [00:00<?, ?it/s]
Calculate teacher chan

RecursionError: maximum recursion depth exceeded in comparison